# Faza 0: Nalaganje podatkov in vzorčenje

**Zastavljen cilj**: Pripravili obvladljivo in čisto začetno bazo iz izhodiščne ~1,2 GB datoteke `accepted_2007_to_2018Q4.csv`. Obdržali smo ločena popolnoma poplačana in neplačana posojila, izbrali bistvene atribute ter kreirali stratificiran vzorec 20.000 vrstic za hitrejše nadaljnje modeliranje.

## 1. Uvoz knjižnic

Najprej smo uvozili vse osnovne knjižnice, ki so bile potrebne za operacije nad podatkovnimi okviri in za razdelitev podatkov.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# TODO (ekipa): Uvozi še druge knjižnice, ki so / bodo prišle prav v tej fazi.

## 2. Nalaganje podatkov

Zaradi velikosti izvorne datoteke smo zagotovili uporabo knjižnice `pandas` s parametrom `low_memory=False`. Ta parameter nam je omogočil varno prebiranje in preprečil prekinitve ob morebitnih opozorilih o mešanih podatkovnih tipih. Začasno smo tudi testirali nalaganje prvih nekaj tisoč vrstic preden smo pospešeno naložili celoto.

In [ ]:
# TODO (ekipa): Preberi CSV datoteko `../data/raw/accepted_2007_to_2018Q4.csv` v nov podatkovni okvir z imenom 'df'
# TODO (ekipa): Prikaži osnovno obliko (npr. z `df.shape`) in izpiši prvih nekaj vrstic.

## 3. Filtriranje in kodiranje ciljne spremenljivke (`loan_status`)

Ker v naši raziskavi obravnavamo binarni klasifikacijski problem napovedovanja neplačila, smo preiskali obsežen originalni seznam statusov:

1. **Obdržali** smo le tiste vrstice, pri katerih je bil proces posojila dokončno in uradno zaključen (status `'Fully Paid'` ali `'Charged Off'`).
2. **Kodirali** smo poenostavljeno ciljno spremenljivko na način, da je uspešen konec `'Fully Paid'` predstavljen z vrednostjo `0`, ter neuspeh `'Charged Off'` z vrednostjo `1`. Vse nadaljnje analize (zlasti XGboost in SHAP) to sedaj tolmačijo kot uresničenje nezaželenega dogodka (ang. *default*).

In [ ]:
# TODO (ekipa): S pomočjo df['loan_status'].isin(...) obdrži samo vrstice za naštete dokončane kredite.
# TODO (ekipa): Preslikaj besedilne statuse (npr. z .map({'Fully Paid':0, 'Charged Off':1})) v navadne inte (0 in 1).
# TODO (ekipa): Prikaži, da je bilo uspešno, in preveri vsebino z `df['loan_status'].value_counts()`.

## 4. Izbira spremenljivk

Sam izvirni nabor zajema več kot 150 stolpcev. Ker številni navajajo podatke, ki nam na točkovni začetni odobritvi posojila ne bi bili javno na voljo (t.i. uhajanje podatkov iz prihodnosti ali *Data Leakage*), ali pa so na spisku delovali popolnoma nerelevantno oziroma prazno, smo zadržali zgolj specifično in vnaprej izbrano podmnožico.

**Ohranili smo naslednje specifične stolpce:**
`loan_amnt`, `funded_amnt`, `term`, `int_rate`, `installment`, `grade`, `sub_grade`, `emp_title`, `emp_length`, `home_ownership`, `annual_inc`, `verification_status`, `desc`, `purpose`, `dti`, `delinq_2yrs`, `earliest_cr_line`, `fico_range_low`, `fico_range_high`, `inq_last_6mths`, `mths_since_last_delinq`, `open_acc`, `pub_rec`, `revol_bal`, `revol_util`, `total_acc`, `application_type`, `tot_cur_bal`, `total_rev_hi_lim`, `acc_open_past_24mths`, `avg_cur_bal`, `bc_open_to_buy`, `bc_util`, `chargeoff_within_12_mths`, `delinq_amnt`, `mo_sin_old_il_acct`, `mo_sin_old_rev_tl_op`, `mo_sin_rcnt_rev_tl_op`, `mo_sin_rcnt_tl`, `mort_acc`, `num_accts_ever_120_pd`, `percent_bc_gt75`, `pub_rec_bankruptcies`, `tax_liens`, `loan_status`

*(Opis/opozorilo: Stolpec za zvezno državo (`addr_state`) je v skladu z dogovorom namerno izključen, saj preveč poslabša the dimenzionalnost naše enačbe s kar 50 novimi stolpci po kodiranju, medtem ko obenem prinaša le skromno informacijsko vrednost.)*


In [ ]:
# TODO (ekipa): Predaj dogovorjeni seznam obdržanih imen (iz zgoraj) v obliki string liste.
# TODO (ekipa): Prepiši in zmanjšaj trenutno tabelo: npr. df = df[columns_to_keep].
# TODO (ekipa): Poglej prečiščene stolpce z `df.info()` ali z izpisom `.columns`.

## 5. Stratificirano vzorčenje na n=20.000

Zaradi smotrno skrajšanih časov učenja in lažjega reševanja NLP problema ob koncu (TF-IDF zamenjava na opisih), smo obširno polno množico skrčili točno na predstavitveni podvzorec obsega 20.000 inštanc.

Pozornost pri tem se je najbolj namenila **stratifikaciji**, in sicer z uporabo parametra ali ustreznega grupiranja nad `loan_status`. Tako smo zmanjšali podatke na številko 20.000, pri čemer je sedaj delež problematičnih (1) ter normalnih (0) takoj udeležen identično kakor pri polnih vseh ~1,4 miljonih podatkih.

In [ ]:
# TODO (ekipa): Pripravi in uporabi algoritem podvzorčenja s stratifikacijo. (Npr. groupby in frac ali train_test_split s fiksnim train_size=20000/len(df) ). Določi seme random_state=42.
# TODO (ekipa): Razmerje razredov ustrezno preveri z izpisom (npr. s `normalize=True`).
# TODO (ekipa): Ne pozabite shraniti končen ekstrahiran podvzorec `../data/lending_club_20k.csv` (npr. z df.to_csv in index=False).

## 6. Delitev na učno in testno množico

Tako dobljen in prečiščen, a sedaj zmanjšan stratificirani nabor (20.000 instanc), je na konec bil strogo razčlenjen v izbranem razmerju **80 proti 20 (80 : 20)** med dve popolnoma neodvisni množici: *Train* in *Test*.

Tudi pri tem ultimatem pomembnem deju se je upoštevalo vzdrževanje omenjenih enakovrebih porazdelitev (ponovna stratifikacija nad statusi posojila).

In [ ]:
# TODO (ekipa): Izvedi razdelitev prejšnjih 20k na 80/20. Z uporabo `train_test_split(df_sample, test_size=0.2, stratify=df_sample['loan_status'], random_state=42)`.
# TODO (ekipa): Nauči in shrani dobljeno ločeno učno podmnožico v datoteki `../data/train_raw.csv` z index=False.
# TODO (ekipa): Izvozi testno podmnožico v posamezno `../data/test_raw.csv` (index=False).
# TODO (ekipa): Natisni velikosti teh polj, da razjasni dimenzijo na samem koncu Notebooka.